# Baseline AI hình ảnh xe

Chạy lần lượt các cell trong Colab/Kaggle với GPU. Dữ liệu hiện tại chỉ có validation nội bộ; không dùng điểm này để cam kết chất lượng thực tế. OCR do module khác xử lý. Gói ZIP đã giữ license CarDD và các nhãn nguồn chưa xác nhận.

Không có bước tự upload dữ liệu hoặc mua GPU. Đặt ZIP trong Drive riêng của bạn hoặc dùng private Kaggle input. Python 3.10–3.12 được khuyến nghị.

Nếu cell cài dependencies yêu cầu restart, chọn Restart session/runtime (không xóa runtime), chạy lại cell đường dẫn rồi bỏ qua cell cài. Python 3.13 cần NumPy >=2.1.


In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile

# Colab: đặt cloud-baseline-v2.zip trong MyDrive; Kaggle: sửa thành đường dẫn input riêng.
ZIP_PATH = Path("/content/drive/MyDrive/cloud-baseline-v2.zip")
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
DATA = WORK / "vision-baseline"
RUNS = WORK / "runs-vision-v1"  # đổi tên khi chạy lại; script không ghi đè kết quả.
EPOCHS = 40
BATCH = 16  # giảm còn 8 nếu GPU thiếu VRAM.


In [ ]:
if not ZIP_PATH.is_file() and str(ZIP_PATH).startswith("/content/drive/"):
    from google.colab import drive
    drive.mount("/content/drive")
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Đặt ZIP_PATH tới gói dữ liệu đã chuẩn bị: {ZIP_PATH}")
WORK.mkdir(parents=True, exist_ok=True)
if not DATA.exists():
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for item in archive.infolist():
            target = (WORK / item.filename).resolve()
            if not target.is_relative_to(WORK.resolve()):
                raise ValueError("Đường dẫn ZIP nằm ngoài thư mục làm việc")
        archive.extractall(WORK)
print(json.loads((DATA / "bundle-manifest.json").read_text()))


In [ ]:
# ZIP cũ có thể còn ép numpy<2, không tương thích Python 3.13.
requirements = (DATA / "requirements-vision-train.txt").read_text()
requirements = requirements.replace(
    "numpy>=1.26,<2\n",
    'numpy>=1.26,<2; python_version < "3.13"\n'
    'numpy>=2.1,<3; python_version >= "3.13"\n',
)
notebook_requirements = DATA / "requirements-vision-notebook.txt"
notebook_requirements.write_text(requirements)
numpy_already_loaded = "numpy" in sys.modules
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(notebook_requirements)], check=True)
# Kiểm tra bằng process mới, không dùng thư viện còn giữ trong kernel.
subprocess.run([sys.executable, "-c", (
    "import numpy, numpy.random, torch, cv2, sklearn; "
    "print('NumPy:', numpy.__version__); "
    "assert torch.cuda.is_available(), 'Bật GPU trong Runtime/Accelerator'; "
    "print('GPU:', torch.cuda.get_device_name(0))"
)], check=True)
if numpy_already_loaded:
    raise RuntimeError(
        "Dependencies đã cài. Restart session/runtime, chạy lại cell đường dẫn, "
        "rồi tiếp tục từ preflight hoặc cell dự đoán. Không chạy lại cell cài này."
    )
print("Dependencies đã kiểm tra; nếu gặp NumPy ABI ở kernel, restart trước khi dự đoán.")


## Nhận dạng mẫu xe VinFast — YOLO11n Classification

Train 167 ảnh, validation 41 ảnh, 14 nhãn. Nerio Green để riêng chờ bổ sung. YOLO mặc định có bước crop ảnh; cần xem preview để phát hiện crop làm mất đặc điểm xe.

Cell sau kiểm tra dữ liệu mà chưa tải checkpoint hoặc train.

In [ ]:
TASK = "vehicles"
CLI = DATA / "scripts/train_vision_baseline.py"
subprocess.run([sys.executable, str(CLI), "--task", TASK, "--data-root", str(DATA),
                "--check-only"], check=True)


## Train và đánh giá

Cell này tải checkpoint pretrained khi cần và bắt đầu train trên GPU. `best.pt` được chọn theo validation fitness mặc định Ultralytics. Metrics/class maps, hash dữ liệu và checkpoint, package versions được lưu trong `evaluation.json`. Threshold nhận dạng 0.7 và detection 0.25 là ngưỡng thử, chưa hiệu chỉnh bằng dữ liệu thật.

In [ ]:
subprocess.run([sys.executable, str(CLI), "--task", TASK, "--data-root", str(DATA),
                "--output", str(RUNS), "--epochs", str(EPOCHS), "--batch", str(BATCH),
                "--device", "0"], check=True)
ARTIFACTS = RUNS / TASK
report = json.loads((ARTIFACTS / "evaluation.json").read_text())
print(json.dumps(report, ensure_ascii=False, indent=2))


## Xem dự đoán và xuất bằng chứng cho agent

Ví dụ dưới dùng một ảnh validation, không phải ảnh test độc lập. Dòng xe chỉ là đề xuất cần xác nhận. `not_detected` chỉ nghĩa là không tìm thấy bbox đạt ngưỡng. Agent không được tự cập nhật dòng xe đã xác nhận hoặc kết luận chẩn đoán từ JSON này.

In [ ]:
TASK = "vehicles"
ARTIFACTS = RUNS / TASK
report = json.loads((ARTIFACTS / "evaluation.json").read_text())
sys.path.insert(0, str(DATA))
from ultralytics import YOLO
from src.services.vision.contract import VisionEvidence, vehicle_result, damage_result
from IPython.display import display, Image as DisplayImage
best = YOLO(str(ARTIFACTS / "weights/best.pt"))
sample = sorted((DATA / "vehicles/val").glob("*/*.jpg"))[0]
result = best.predict(str(sample), device=0, conf=0.25, verbose=False)[0]
version = report["checkpoint_sha256"]
scores = [(result.names[i], float(v)) for i, v in enumerate(result.probs.data.cpu().tolist())]
evidence = VisionEvidence(image_id=sample.stem, vehicle=vehicle_result(scores, version))
preview = ARTIFACTS / "prediction-example.jpg"
result.save(filename=str(preview))
display(DisplayImage(filename=str(preview)))
(ARTIFACTS / "prediction-example.json").write_text(evidence.model_dump_json(indent=2))
print(evidence.model_dump_json(indent=2))


## Lưu kết quả trước khi notebook hết phiên

ZIP chứa weights, metrics, confusion matrix/plots và ví dụ JSON. Colab tải qua cell bên dưới; Kaggle lưu file ZIP trong Output. Với run dài, nên đặt `RUNS` trên Drive để checkpoint được giữ giữa các phiên.

In [ ]:
import shutil
artifact_zip = shutil.make_archive(str(WORK / (TASK + "-artifacts")), "zip", ARTIFACTS)
print(artifact_zip)
# Colab: bỏ comment hai dòng sau để tải về máy.
# from google.colab import files
# files.download(artifact_zip)
